# Portkey AI Gateway 

**Portkey's AI Gateway** is a fast, reliable proxy that sits between your app and 250+ LLM providers. It gives you a **unified API** with production-grade features built in — no code changes needed when switching providers.

### What we'll cover today

| # | Feature | Why it matters |
|---|---------|----------------|
| 1 | **Setup** | One SDK, one API key |
| 2 | **Chat Completions** | OpenAI-compatible interface for any provider |
| 3 | **Fallbacks** | Auto-switch to backup LLMs on failure |
| 4 | **Automatic Retries** | Exponential backoff with configurable triggers |
| 5 | **Caching** | Serve repeated requests 20x faster |
| 6 | **Load Balancing** | Distribute traffic across models/keys |
| 7 | **Conditional Routing** | Route by metadata, params, or custom logic |
| 8 | **Canary Testing** | Safely test new models on a % of traffic |
| 9 | **Streaming** | Real-time token streaming with all features intact |
| 10 | **Guardrails** | Input/output validation at the gateway edge |
| 11 | **Metadata & Observability** | Tag, trace, and monitor every request |
| 12 | **Nested Strategies** | Compose fallback + LB + retry + cache |

>
> **Prerequisites:** Python 3.9+, a [Portkey account](https://app.portkey.ai), and a virtual key configured for your LLM provider

---
## 1. Setup

Install the SDK and configure your API key. That's it — one dependency.

In [1]:
%pip install portkey-ai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import json
from dotenv import load_dotenv
from portkey_ai import Portkey

load_dotenv()

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

# ----- Replace these with your own virtual key slugs -----
# Virtual keys wrap your provider credentials (OpenAI, Anthropic, etc.)
# Create them at: https://app.portkey.ai/virtual-keys
# VIRTUAL_KEY_SLUG = "@your-virtual-key"  # e.g. "@my-openai" or "@hk-openrouter"
PRIMARY_MODEL = "@anthropic/claude-sonnet-4-6"   # change to your model
PRIMARY_MODEL_1 = "@hk-gemini-demo/gemini-2.5-flash-001"
BACKUP_MODEL = "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"  # fallback model
# ---------------------------------------------------------

if not PORTKEY_API_KEY:
    raise EnvironmentError(
        "PORTKEY_API_KEY not found. "
        "Create a .env file with: PORTKEY_API_KEY=your_key_here"
    )

masked = PORTKEY_API_KEY[:8] + "*" * 20
print(f"PORTKEY_API_KEY: {masked}")
print(f"Primary model:  {PRIMARY_MODEL}")
print(f"Backup model:   {BACKUP_MODEL}")
print("\nReady to go!")

PORTKEY_API_KEY: w1vVZCBe********************
Primary model:  @anthropic/claude-sonnet-4-6
Backup model:   @hk-openrouter/meta-llama/llama-3.3-70b-instruct

Ready to go!


---
## 2. Basic Chat Completion

The Portkey SDK is **drop-in compatible with OpenAI's API**. Swap `openai.OpenAI()` for `Portkey()` and everything works — but now you get observability, caching, and routing for free.

```
Your App  ──▶  Portkey Gateway  ──▶  OpenAI / Anthropic / Gemini / ...
                   │
                   ├── Logs & Analytics
                   ├── Caching
                   └── Guardrails
```

In [43]:
portkey = Portkey(api_key=PORTKEY_API_KEY)
try :
    response = portkey.chat.completions.create(
        model=PRIMARY_MODEL_1,
        messages=[
            {"role": "system", "content": "You are a concise assistant. Reply in 1-2 sentences."},
            {"role": "user",   "content": "What is an AI Gateway and why do I need one?"},
        ],
        max_tokens=150,
    )
    
    print(f"Model:    {response.model}")
    print(f"Tokens:   {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print("Error Occured")

Error Occured


---
## 3. Fallback Strategy

**Problem:** Your primary LLM goes down or hits rate limits.  
**Solution:** Portkey automatically falls through to backup models — zero downtime for your users.

```
Request ──▶ Model A (primary)
               │ fails
               ▼
            Model B (backup)
               │ fails
               ▼
            Model C (last resort)
```

You can also trigger fallbacks only on **specific status codes** (e.g., `429` for rate limits).

In [6]:
# Basic fallback: if primary fails, try backup
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": PRIMARY_MODEL_1}},
        {"override_params": {"model": BACKUP_MODEL}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)

response = client.chat.completions.create(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50,
)

print(f"Model used: {response.model}")
print(f"Response:   {response.choices[0].message.content}")
print("\n(If primary was down, this would have transparently used the backup.)")

Model used: meta-llama/llama-3.3-70b-instruct
Response:   Hello, it's nice to meet you and I'm here to help answer any questions or provide assistance you may need.

(If primary was down, this would have transparently used the backup.)


In [ ]:
# Fallback only on rate-limit (429)
rate_limit_fallback = {
    "strategy": {"mode": "fallback", "on_status_codes": [429]},
    "targets": [
        {"override_params": {"model": PRIMARY_MODEL}},
        {"override_params": {"model": BACKUP_MODEL}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=rate_limit_fallback)

response = client.chat.completions.create(
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
    max_tokens=50,
)

print(f"Model used: {response.model}")
print(f"Response:   {response.choices[0].message.content}")
print("\n(Only switches on 429 — other errors propagate normally.)")

---
## 4. Automatic Retries

Transient failures happen. Portkey retries with **exponential backoff** so you don't have to build retry logic.

| Setting | Default |
|---|---|
| Max attempts | 5 |
| Retry codes | `429, 500, 502, 503, 504` |
| Backoff | `1s → 2s → 4s → 8s → 16s` |
| Retry-After | Respects provider's `Retry-After` header |

In [ ]:
retry_config = {
    "retry": {"attempts": 3, "on_status_codes": [429, 500, 503]},
    "override_params": {"model": PRIMARY_MODEL},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=retry_config)

response = client.chat.completions.create(
    messages=[{"role": "user", "content": "Say 'retry works!'"}],
    max_tokens=20,
)

print(f"Response: {response.choices[0].message.content}")
print("\nIf the provider returned 429/500/503, Portkey would have")
print("automatically retried up to 3 times with exponential backoff.")

Response: retry works!

If the provider returned 429/500/503, Portkey would have
automatically retried up to 3 times with exponential backoff.


---
## 5. Caching

Serve identical requests **20x faster** and cheaper by caching at the gateway layer.

| Mode | How it works |
|------|-------------|
| **Simple** | Exact match on the full input |
| **Semantic** | Cosine-similarity matching ("similar enough" = cache hit) |

Let's see the speedup in real-time:

In [7]:
cache_config = {
    "cache": {"mode": "simple", "max_age": 60},
    "override_params": {"model": PRIMARY_MODEL},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)
test_msg = [{"role": "user", "content": "What is the capital of France? One word."}]

# Request 1: Cache MISS (goes to the LLM)
t0 = time.time()
resp1 = client.chat.completions.create(messages=test_msg, max_tokens=10)
t1 = time.time() - t0

# Request 2: Cache HIT (served from gateway)
t0 = time.time()
resp2 = client.chat.completions.create(messages=test_msg, max_tokens=10)
t2 = time.time() - t0

print(f"Request 1 (cache miss): {resp1.choices[0].message.content.strip():>10}  [{t1:.3f}s]")
print(f"Request 2 (cache hit):  {resp2.choices[0].message.content.strip():>10}  [{t2:.3f}s]")
print(f"\nSpeedup: {t1/t2:.1f}x faster on cache hit" if t2 < t1 else "\n(Network conditions may mask the speedup)")

Request 1 (cache miss):      Paris  [3.410s]
Request 2 (cache hit):   **Paris**  [1.565s]

Speedup: 2.2x faster on cache hit


In [8]:
# Force refresh: bypass cache and fetch a fresh response
t0 = time.time()
resp3 = client.with_options(cache_force_refresh=True).chat.completions.create(
    messages=test_msg, max_tokens=10,
)
t3 = time.time() - t0

print(f"Force refresh: {resp3.choices[0].message.content.strip():>10}  [{t3:.3f}s]")
print("(Bypassed cache — fetched fresh from the LLM)")

Force refresh:      Paris  [2.264s]
(Bypassed cache — fetched fresh from the LLM)


In [9]:
# Cache namespaces: isolate caches per user/session
msg = [{"role": "user", "content": "What is 7 * 8? Just the number."}]

resp_alice = client.with_options(cache_namespace="user-alice").chat.completions.create(
    messages=msg, max_tokens=10,
)
resp_bob = client.with_options(cache_namespace="user-bob").chat.completions.create(
    messages=msg, max_tokens=10,
)

print(f"Alice's cache: {resp_alice.choices[0].message.content.strip()}")
print(f"Bob's cache:   {resp_bob.choices[0].message.content.strip()}")
print("\nEach namespace is an independent cache partition.")

Alice's cache: 56
Bob's cache:   56

Each namespace is an independent cache partition.


---
## 6. Load Balancing

Distribute traffic across multiple models or API keys with configurable weights.

```
             ┌─── 70% ──▶ Model A
Request ─────┤
             └─── 30% ──▶ Model B
```

**Use cases:**
- Spread rate limits across multiple API keys
- Cost optimization (route most traffic to cheaper models)
- A/B testing different models

In [10]:
lb_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": PRIMARY_MODEL},  "weight": 0.7},
        {"override_params": {"model": BACKUP_MODEL}, "weight": 0.3},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=lb_config)

model_counts = {}
N = 10

for i in range(N):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Say 'hello {i}'."}],
        max_tokens=10,
    )
    model_counts[resp.model] = model_counts.get(resp.model, 0) + 1

print(f"Distribution over {N} requests:")
for model, count in sorted(model_counts.items(), key=lambda x: -x[1]):
    pct = count / N * 100
    bar = '#' * int(pct / 5)
    print(f"  {model:<50} {count:>2} ({pct:.0f}%) {bar}")

Distribution over 10 requests:
  meta-llama/llama-3.3-70b-instruct                   5 (50%) ##########
  claude-sonnet-4-6                                   5 (50%) ##########


---
## 7. Conditional Routing

Route requests to different models based on **runtime conditions** evaluated at the gateway edge.

| Condition Source | Example |
|---|---|
| `metadata.<key>` | Route paid users to GPT-4, free users to GPT-3.5 |
| `params.model` | Use model aliases ("fast", "smart") |
| `params.temperature` | High-temp requests → creative model |

**Operators:** `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$regex`, `$and`, `$or`

In [ ]:
# Route based on user plan: paid → premium model, free → basic model
routing_config = {
    "strategy": {
        "mode": "conditional",
        "conditions": [
            {
                "query": {"metadata.user_plan": {"$eq": "paid"}},
                "then": "premium-target",
            },
            {
                "query": {"metadata.user_plan": {"$eq": "free"}},
                "then": "basic-target",
            },
        ],
        "default": "basic-target",
    },
    "targets": [
        {"name": "premium-target", "override_params": {"model": PRIMARY_MODEL}},
        {"name": "basic-target",   "override_params": {"model": BACKUP_MODEL}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=routing_config)

for plan in ["paid", "free"]:
    resp = client.with_options(
        metadata={"user_plan": plan}
    ).chat.completions.create(
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=10,
    )
    print(f"  user_plan={plan!r:>6}  →  routed to: {resp.model}")

In [ ]:
# Model aliases: "fast" and "smart" map to different backends
alias_config = {
    "strategy": {
        "mode": "conditional",
        "conditions": [
            {"query": {"params.model": {"$eq": "fast"}},  "then": "fast-target"},
            {"query": {"params.model": {"$eq": "smart"}}, "then": "smart-target"},
        ],
        "default": "fast-target",
    },
    "targets": [
        {"name": "fast-target",  "override_params": {"model": BACKUP_MODEL}},
        {"name": "smart-target", "override_params": {"model": PRIMARY_MODEL}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=alias_config)

for alias in ["fast", "smart", "unknown"]:
    resp = client.chat.completions.create(
        model=alias,
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=10,
    )
    print(f"  model={alias!r:>10}  →  routed to: {resp.model}")

---
## 8. Canary Testing

Safely test a new model in production by sending only **5% of traffic** to it.

```
             ┌─── 95% ──▶ Proven Model (production)
Request ─────┤
             └───  5% ──▶ New Model (canary)
```

Monitor the canary via Portkey's analytics dashboard — compare cost, latency, error rates, and user feedback before rolling out fully.

In [11]:
canary_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": PRIMARY_MODEL},  "weight": 0.95},
        {"override_params": {"model": BACKUP_MODEL}, "weight": 0.05},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=canary_config)

model_counts = {}
N = 20

for i in range(N):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Test request {i}"}],
        max_tokens=5,
    )
    model_counts[resp.model] = model_counts.get(resp.model, 0) + 1

print(f"Canary distribution over {N} requests:")
for model, count in sorted(model_counts.items(), key=lambda x: -x[1]):
    pct = count / N * 100
    print(f"  {model:<50} {count:>2} ({pct:.0f}%)")

Canary distribution over 20 requests:
  claude-sonnet-4-6                                  18 (90%)
  meta-llama/llama-3.3-70b-instruct                   2 (10%)


---
## 9. Streaming

Stream tokens in real-time — all gateway features (fallback, cache, guardrails) work with streaming too.

In [37]:
portkey = Portkey(api_key=PORTKEY_API_KEY)

print("Streaming response:\n")

stream = portkey.chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Explain why AI Gateways matter in 3 bullet points."},
    ],
    max_tokens=200,
    stream=True,
)

full_response = ""
for chunk in stream:
    delta = chunk.choices
    if delta:
        print(delta[0].delta.content, end="", flush=True)
        full_response += str(delta[0].delta.content)

print("\n\n--- Stream complete ---")

Streaming response:

None• **Security & Control** – AI Gateways act as a centralized layer to enforce authentication, rate limiting, and content filtering, preventing misuse, data leaks, and unauthorized access to AI models.

• **Observability & Governance** – They provide unified logging, monitoring, and auditing across all AI interactions, giving organizations visibility into usage, costs, and compliance adherence.

• **Flexibility & Reliability** – They enable seamless routing between multiple AI providers, models, or versions, reducing vendor lock-in and ensuring fallback options when a model fails or degrades.None

--- Stream complete ---


In [39]:
# Streaming with fallback — if primary fails mid-stream, backup kicks in
stream_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": PRIMARY_MODEL}},
        {"override_params": {"model": BACKUP_MODEL}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=stream_fallback_config)

print("Streaming with fallback:\n")

stream = client.chat.completions.create(
    messages=[{"role": "user", "content": "Count from 1 to 10."}],
    max_tokens=100,
    stream=True,
)

full_response = ""
for chunk in stream:
    delta = chunk.choices
    if delta:
        print(delta[0].delta.content, end="", flush=True)
        full_response += str(delta[0].delta.content)

print("\n\n--- Stream complete ---")

Streaming with fallback:

NoneHere is counting from 1 to 10:

1, 2, 3, 4, 5, 6, 7, 8, 9, 10None

--- Stream complete ---


---
## 10. Guardrails

Validate inputs **before** they reach the LLM, or validate outputs **before** they reach your user.

| Hook | When it runs |
|------|-------------|
| `before_request_hooks` | Before sending to the LLM |
| `after_request_hooks` | After receiving the LLM response |

| Verdict | `deny=true` | `deny=false` |
|---------|-------------|---------------|
| PASS | 200 (normal) | 200 (normal) |
| FAIL | **446** (blocked) | **246** (soft fail, logged) |

Below we use an inline regex guardrail to block emails in user input.

In [40]:
EMAIL_REGEX = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

guardrail_config = {
    "strategy": {"mode": "single"},
    "targets": [{"override_params": {"model": PRIMARY_MODEL}}],
    "before_request_hooks": [
        {
            "type": "guardrail",
            "id": "email_blocker",
            "checks": [
                {
                    "id": "default.regexMatch",
                    "parameters": {"rule": EMAIL_REGEX},
                }
            ],
            "deny": "true",
        }
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=guardrail_config)

# Test 1: Input WITH an email — should be BLOCKED
print("Test 1: Input with email address")
print("-" * 40)
try:
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": "Send a mail to test@example.com"}],
    )
    print(f"  Passed (unexpected): {resp.choices[0].message.content[:80]}")
except Exception as e:
    print(f"  BLOCKED by guardrail (expected)")
    print(f"  Error: {str(e)[:120]}")

print()

# Test 2: Clean input — should PASS
print("Test 2: Clean input (no email)")
print("-" * 40)
try:
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": "Explain quantum computing in one sentence."}],
    )
    print(f"  Passed (expected): {resp.choices[0].message.content[:120]}")
except Exception as e:
    print(f"  Blocked (unexpected): {str(e)[:120]}")

Test 1: Input with email address
----------------------------------------
  Passed (unexpected): I'm sorry, but I **cannot send emails**. I'm a text-based AI assistant and I don

Test 2: Clean input (no email)
----------------------------------------
  Passed (expected): Quantum computing uses quantum mechanical phenomena like superposition and entanglement to process information in ways t


---
## 11. Metadata & Observability

Tag every request with custom metadata for filtering, grouping, and debugging in the Portkey dashboard.

- **`trace_id`** — Correlate related requests across a user session or pipeline
- **`metadata`** — Arbitrary key-value pairs (user ID, environment, feature flag, etc.)
- All metadata is searchable and filterable in [Portkey Logs](https://app.portkey.ai/logs)

In [41]:
import uuid

portkey = Portkey(api_key=PORTKEY_API_KEY)

trace_id = f"demo-{uuid.uuid4().hex[:8]}"

response = portkey.with_options(
    trace_id=trace_id,
    metadata={
        "user_id": "user-42",
        "environment": "demo",
        "feature": "onboarding-flow",
        "session_id": "sess-abc123",
    },
).chat.completions.create(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "What is Portkey?"}],
    max_tokens=100,
)

print(f"Trace ID:  {trace_id}")
print(f"Model:     {response.model}")
print(f"Response:  {response.choices[0].message.content[:150]}")
print(f"\nFind this request in the Portkey dashboard by searching for trace_id: {trace_id}")

Trace ID:  demo-48d71ca9
Model:     claude-sonnet-4-6
Response:  # Portkey

Portkey could refer to several things depending on the context:

## Harry Potter Universe
The most well-known reference — a **Portkey** is 

Find this request in the Portkey dashboard by searching for trace_id: demo-48d71ca9


---
## 12. Nested Strategies — The Full Picture

All strategies are **composable**. Here's a production-ready config combining everything:

```
┌─────────────────────────────────────────────────────┐
│                    CACHE (60s)                      │
│  ┌───────────────────────────────────────────────┐  │
│  │              RETRY (3 attempts)               │  │
│  │  ┌─────────────────────────────────────────┐  │  │
│  │  │            FALLBACK                     │  │  │
│  │  │  ┌─────────────────────────────────┐    │  │  │
│  │  │  │ Target 1: Primary Model         │    │  │  │
│  │  │  │         (50% weight)             │    │  │  │
│  │  │  └─────────────────────────────────┘    │  │  │
│  │  │  ┌─────────────────────────────────┐    │  │  │
│  │  │  │ Target 2: Load Balanced Group   │    │  │  │
│  │  │  │  ├── Model B (fallback primary) │    │  │  │
│  │  │  │  └── Model C (fallback backup)  │    │  │  │
│  │  │  │         (50% weight)             │    │  │  │
│  │  │  └─────────────────────────────────┘    │  │  │
│  │  └─────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

In [ ]:
production_config = {
    "cache": {"mode": "simple", "max_age": 60},
    "retry": {"attempts": 3, "on_status_codes": [429, 500, 503]},
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "override_params": {"model": PRIMARY_MODEL},
        },
        {
            "override_params": {"model": BACKUP_MODEL},
        },
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=production_config)

response = client.chat.completions.create(
    messages=[{"role": "user", "content": "What is 1+1? Answer with just the number."}],
    max_tokens=10,
)

print(f"Model:    {response.model}")
print(f"Response: {response.choices[0].message.content}")
print()
print("Active layers:")
print("  - Cache:    simple, 60s TTL")
print("  - Retry:    3 attempts on 429/500/503")
print("  - Fallback: primary → backup")
print("\nAll of this is config-driven. No code changes needed.")

In [ ]:
# Load Balanced targets with nested fallback
nested_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {
            "override_params": {"model": PRIMARY_MODEL},
            "weight": 0.5,
        },
        {
            "strategy": {"mode": "fallback", "on_status_codes": [429]},
            "targets": [
                {"override_params": {"model": BACKUP_MODEL}},
                {"override_params": {"model": PRIMARY_MODEL}},
            ],
            "weight": 0.5,
        },
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=nested_config)

model_counts = {}
for i in range(6):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Nested test {i}"}],
        max_tokens=10,
    )
    model_counts[resp.model] = model_counts.get(resp.model, 0) + 1
    print(f"  Request {i+1}: {resp.model}")

print(f"\nDistribution: {model_counts}")
print("(50% direct to primary, 50% to fallback group)")

---
## Recap

Everything we just demoed is:

- **Config-driven** — change routing, caching, and guardrails without redeploying
- **Provider-agnostic** — same code works with OpenAI, Anthropic, Google, Mistral, local models, and 250+ more
- **OpenAI-compatible** — drop-in replacement, no new APIs to learn
- **Observable** — every request is logged with metadata, cost, latency, and token usage

### What's on the Portkey Dashboard

| Feature | What you see |
|---|---|
| **Logs** | Every request with full input/output, latency, cost, status |
| **Analytics** | Aggregate views — cost over time, error rates, p50/p95 latency |
| **Traces** | Correlated request chains via `trace_id` |
| **Alerts** | Notifications on error spikes, cost anomalies |

### Next steps

1. **Try it:** [app.portkey.ai](https://app.portkey.ai) — free tier, no credit card
2. **Docs:** [docs.portkey.ai](https://docs.portkey.ai)
3. **Open-source gateway:** [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway) — self-host if you need to
4. **SDK:** `pip install portkey-ai` — Python, Node.js, REST API